In [8]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_links, fetch_website_contents
from IPython.display import Markdown, display,update_display
from openai import OpenAI
import requests
import json
from selenium import webdriver
from google import genai
import tiktoken

In [4]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
google_api_key =  os.getenv('GOOGLE_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
claude_api_key = os.getenv('CLAUDE_API_KEY')

In [5]:
#ENDPOINTS
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"

In [15]:
#MODELS
OPEN_AI_MODEL = 'gpt-5-nano'
GEMINI_MODEL = 'gemini-2.5-flash-preview-tts'
OLLAMA_MODEL = 'gemma3:1b'
CLAUDE_MODEL = 'claude-haiku-4-5@20251001'

In [7]:
#CLIENTS
openai = OpenAI()
genai_client = genai.Client()
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=gemini_api_key)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
claude = OpenAI(base_url=ANTHROPIC_BASE_URL, api_key=claude_api_key)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [10]:
links = fetch_website_links("https://www.nytimes.com/")
links = [link for link in links if link.startswith("https")]

In [11]:
links

['https://www.nytimes.com/es/',
 'https://cn.nytimes.com',
 'https://www.nytimes.com/section/todayspaper',
 'https://www.nytimes.com/2026/02/06/us/politics/trump-schumer-penn-station-dulles-airport-renaming.html',
 'https://www.nytimes.com/2026/02/06/nyregion/gateway-tunnel-project-funding-trump.html',
 'https://www.nytimes.com/2026/02/06/us/politics/trump-schumer-penn-station-dulles-airport-renaming.html',
 'https://www.nytimes.com/2026/02/06/us/politics/hegseth-defense-harvard.html',
 'https://www.nytimes.com/2026/02/06/us/politics/trump-republican-governors-white-house.html',
 'https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html',
 'https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html',
 'https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html',
 'https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html',
 'https://www.nytimes.com/2026/02/06/us/po

In [12]:
#SYSTEM PROMPTS
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [13]:
def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

        """
    links = fetch_website_links(url)
    links = [link for link in links if link.startswith("https")]
    user_prompt += "\n".join(links)
    return user_prompt

In [14]:
display(Markdown(get_links_user_prompt("https://www.nytimes.com/")))


        Here is the list of links on the website https://www.nytimes.com/ -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

        https://www.nytimes.com/es/
https://cn.nytimes.com
https://www.nytimes.com/section/todayspaper
https://www.nytimes.com/2026/02/06/us/politics/trump-schumer-penn-station-dulles-airport-renaming.html
https://www.nytimes.com/2026/02/06/nyregion/gateway-tunnel-project-funding-trump.html
https://www.nytimes.com/2026/02/06/us/politics/trump-schumer-penn-station-dulles-airport-renaming.html
https://www.nytimes.com/2026/02/06/us/politics/hegseth-defense-harvard.html
https://www.nytimes.com/2026/02/06/us/politics/trump-republican-governors-white-house.html
https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html
https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html
https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html
https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html
https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html
https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html
https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html
https://www.nytimes.com/2026/02/06/us/politics/trump-obamas-video-apes-truth-social.html
https://www.nytimes.com/athletic/7028050/2026/02/06/milan-cortina-olympics-opening-ceremony/
https://www.nytimes.com/2026/02/06/world/europe/2026-olympics-one-athlete-countries-malaysia.html
https://www.nytimes.com/athletic/6981233/2026/02/02/winter-olympics-us-american-athletes-to-watch-milan-cortina/
https://www.nytimes.com/athletic/6980958/2026/02/02/winter-olympics-international-athletes-to-watch-milan-cortina/
https://www.nytimes.com/live/2026/02/06/world/milan-winter-olympics-opening-ceremony
https://www.nytimes.com/athletic/7027486/2026/02/06/olympics-figure-skating-alysa-liu-usa/
https://www.nytimes.com/2026/02/06/world/europe/jd-vance-olympics-opening-ceremony.html
https://www.nytimes.com/athletic/7026802/2026/02/06/lindsey-vonn-olympics-downhill-torn-acl-training/
https://www.nytimes.com/2026/02/03/us/savannah-guthrie-mom-missing.html
https://www.nytimes.com/2026/02/05/health/savannah-guthrie-mom-nancy-missing-pacemaker.html
https://www.nytimes.com/2026/02/04/us/savannah-guthrie-plea-statement.html
https://www.nytimes.com/2026/02/03/us/savannah-guthrie-mom-nancy-guthrie-search-updates.html
https://www.nytimes.com/2026/02/06/us/savannah-nancy-guthrie-missing-updates.html
https://www.nytimes.com/2026/02/06/us/famous-kidnapping-cases-abduction-numbers.html
https://www.nytimes.com/2026/02/06/us/savannah-nancy-guthrie-missing-updates.html
https://www.nytimes.com/tips
https://www.nytimes.com/2026/02/06/books/mass-market-paperback-books.html
https://www.nytimes.com/2026/02/06/magazine/trump-miller-immigration-ice.html
https://www.nytimes.com/2026/02/05/us/minnesota-el-paso-ice-immigration.html
https://www.nytimes.com/2026/02/05/us/politics/senate-immigration-enforcement-dhs-shutdown.html
https://www.nytimes.com/2026/02/05/us/minnesota-ice-agents-deployment.html
https://www.nytimes.com/2026/02/03/us/politics/trump-ice-immigration-polls-republicans.html
https://www.nytimes.com/2026/02/06/us/us-deportation-liam-conejo-ramos-minnesota.html
https://www.nytimes.com/2026/02/06/us/politics/cuban-deportees-guantanamo.html
https://www.nytimes.com/2026/02/06/us/us-deportation-liam-conejo-ramos-minnesota.html
https://www.nytimes.com/2026/01/30/us/what-to-know-epstein-files.html
https://www.nytimes.com/2026/01/31/us/epstein-trump-gates-musk-tisch-andrew.html
https://www.nytimes.com/2026/02/01/us/trump-epstein-files.html
https://www.nytimes.com/2026/02/05/world/europe/epstein-royal-friends-prince-andrew-sarah-ferguson-mette-marit.html
https://www.nytimes.com/2026/02/03/nyregion/epstein-files-victim-names.html
https://www.nytimes.com/2026/02/06/us/leon-botstein-bard-college-epstein.html
https://www.nytimes.com/2026/02/06/business/brad-karp-paul-weiss-epstein.html
https://www.nytimes.com/2026/02/06/us/leon-botstein-bard-college-epstein.html
https://www.nytimes.com/2026/02/06/world/asia/china-canada-death-sentence-overturned.html
https://www.nytimes.com/2026/02/06/world/europe/us-iran-talks-oman.html
https://www.nytimes.com/2026/02/05/world/middleeast/iran-protests-crackdown.html
https://www.nytimes.com/2026/01/25/world/middleeast/iran-how-crackdown-was-done.html
https://www.nytimes.com/2026/02/06/world/europe/us-iran-talks-oman.html
https://www.nytimes.com/2026/02/06/us/politics/trump-iran-nuclear.html
https://www.nytimes.com/2026/02/06/us/politics/trump-iran-nuclear.html
https://www.nytimes.com/2026/02/06/nyregion/new-jersey-special-election-malinowski-mejia.html
https://www.nytimes.com/2026/02/06/nyregion/new-york-prison-body-scanners-women.html
https://www.nytimes.com/2026/02/06/arts/cassens-menton-france-museum-cocteau.html
https://www.nytimes.com/2026/02/06/arts/cassens-menton-france-museum-cocteau.html
https://www.nytimes.com/2026/02/06/nyregion/duran-nypd-bronx-manslaughter-cooler.html
https://www.nytimes.com/athletic/6997977/2026/01/25/super-bowl-2026-preview-odds-patriots-seahawks-analysis/
https://www.nytimes.com/athletic/6997991/2026/01/26/super-bowl-patriots-seahawks-bad-bunny-pro-bowl/
https://www.nytimes.com/athletic/7008644/2026/02/01/super-bowl-storylines-patriots-seahawks-bad-bunny/
https://www.nytimes.com/athletic/7025357/2026/02/05/bad-bunny-super-bowl-halftime-show-grammy-awards/
https://www.nytimes.com/2026/02/06/arts/television/super-bowl-ads-ranked-2026.html
https://www.nytimes.com/2026/02/06/us/san-francisco-super-bowl.html
https://www.nytimes.com/section/opinion
https://www.nytimes.com/2026/02/06/opinion/capitalism-industry-financialization.html
https://www.nytimes.com/2026/02/06/opinion/ezra-klein-podcast-doctorow-wu.html
https://www.nytimes.com/2026/02/06/opinion/olympics-america-pride.html
https://www.nytimes.com/2026/02/06/opinion/trump-nuclear-arms-control-proposal.html
https://www.nytimes.com/2026/02/06/briefing/let-the-games-begin.html
https://www.nytimes.com/2026/02/06/business/stock-market-rebound-dow-50000.html
https://www.nytimes.com/2026/02/06/world/americas/haiti-crisis-leadership-gangs.html
https://www.nytimes.com/2026/02/06/arts/music/lil-jon-son-dead.html
https://www.nytimes.com/2026/02/06/world/asia/saalumarada-thimmakka-dead.html
https://www.nytimes.com/athletic/
https://www.nytimes.com/athletic/connections-sports-edition
https://cooking.nytimes.com/
https://www.nytimes.com/wirecutter/
https://www.nytimes.com/games/wordle/index.html
https://www.nytimes.com/games/connections?GAMES_connectionsRollout_1130=1_ConnectionsV2
https://www.nytimes.com/puzzles/spelling-bee
https://www.nytimes.com/crosswords/game/daily
https://help.nytimes.com/hc/en-us/articles/115014792127-Copyright-Notice
https://www.nytco.com/
https://help.nytimes.com/hc/en-us/articles/115015385887-Contact-The-New-York-Times
https://help.nytimes.com/hc/en-us/articles/115015727108-Accessibility
https://www.nytco.com/careers/
https://advertising.nytimes.com/
https://advertising.nytimes.com/custom-content/
https://help.nytimes.com/hc/en-us/articles/10940941449492-The-New-York-Times-Company-Privacy-Policy
https://www.nytimes.com/privacy/cookie-policy
https://help.nytimes.com/hc/en-us/articles/115014893428-Terms-of-Service
https://help.nytimes.com/hc/en-us/articles/115014893968-Terms-of-Sale
https://www.nytimes.com/ca/
https://www.nytimes.com/international/
https://help.nytimes.com/hc/en-us
https://www.nytimes.com/subscription?campaignId=37WXW

In [16]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=OPEN_AI_MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
#SYSTEM PROMPTS

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [19]:
#USER PROMPT

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
        You are looking at a company called: {company_name}
        Here are the contents of its landing page and other relevant pages;
        use this information to build a short brochure of the company in markdown without code blocks.\n\n
        """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt